# ハンズオン① LangGraph で ReAct エージェントを作る

**対応セクション**: 3-1「エージェントの構造（LLM + ループ + ツール）」
**推奨所要時間**: 約 75 分
**使用ライブラリ**: LangGraph, LangChain, transformers
**必要環境**: GPU（DGX Spark）

---

> ## このノートブックの結論
>
> **エージェントとは「LLM + ループ + ツール」です。**
> LLM は文章を書くことしかできません。しかし「ツールを呼びたい」と決まった形式で書かせ、
> プログラムがそれを読んで実行し、結果を LLM に返す——これを繰り返すと、LLM は検索も計算もファイル操作もできるようになります。
> 魔法のように見えるエージェントも、中身はこの単純なループです。

---

## このノートブックの目標

1. ツールを持たない LLM が、知らないことや計算を間違えることを確かめる
2. **ツール呼び出しの正体** を、プロンプトとモデルの出力の文字列で確かめる
3. LangGraph の `StateGraph` で、考える（agent）→ 行動する（tools）→ 観察する のループを組み立て、図にする
4. 検索・Python 実行・ファイル読み書きの 3 種類のツールを持つエージェントを動かし、**トレース** で中で何が起きたかを読む

> **モデルについて**: エージェントの頭脳には `Qwen/Qwen2.5-7B-Instruct`（Alibaba 製、約 15 GB）を 4bit で読み込んで使います。
> ツール呼び出しを学習済みで、日本語も扱えるモデルです。第1・2章の Llama-3.2-1B ではツールをうまく選べないため、こちらを使います。

---

# Part 0: セットアップ

In [ ]:
# 実行時間: 数秒
import os
os.environ.setdefault('ANONYMIZED_TELEMETRY', 'False')

import sys
import json
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 第3章の共通部品（chapter3/agent_kit.py）を読み込めるようにする
CH3_DIR = (Path('..') if Path('../agent_kit.py').exists() else Path('chapter3')).resolve()
sys.path.insert(0, str(CH3_DIR))
from agent_kit import LocalChatModel, TraceHandler, parse_tool_calls, run_python as _run_python, workspace_path

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

print('キャッシュ先:', os.environ.get('HF_HOME', '~/.cache/huggingface'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用デバイス: {device}')

# エージェントがファイルを読み書きする作業ディレクトリ（chapter3/workspace）
WORKSPACE = CH3_DIR / 'workspace'
WORKSPACE.mkdir(exist_ok=True)
print(f'作業ディレクトリ: {WORKSPACE}')

In [ ]:
# 実行時間: 約2〜3分
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if device == 'cuda':
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map='auto')
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
model.eval()
print(f'{MODEL_NAME} を読み込みました（モデルのメモリ: {model.get_memory_footprint() / 1024**3:.1f} GB）')

---

# Part 1: ツールを持たない LLM に聞いてみる

2 つの質問をします。1 つはこのハンズオンの決まり、もう 1 つは計算です。

In [ ]:
# 実行時間: 約30秒
def chat(question: str, max_new_tokens: int = 200) -> str:
    """ツールなしで、普通に質問する"""
    prompt = tokenizer.apply_chat_template([{'role': 'user', 'content': question}],
                                           add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)


Q_RULE = 'このハンズオンで、JupyterLab を30分何も実行せずに放置するとどうなりますか？'
Q_CALC = '1 から 1000 までの整数のうち、7 でも 11 でも割り切れない数はいくつありますか？数だけ答えてください。'
for q in [Q_RULE, Q_CALC]:
    print(f'【質問】{q}\n【回答】{chat(q)}\n')

正解は次のとおりです。

- 1 問目: メモリを空けるためにカーネルが自動で止まる（参加者手順書）
- 2 問目: **780**（1000 − 142 − 90 + 12。7 の倍数 142 個、11 の倍数 90 個、77 の倍数 12 個）

LLM は手順書を読んでいないので 1 問目には答えられず、2 問目のような計算は「それらしい数」を書いて間違えがちです。
第2章では RAG で知識を渡しましたが、**何を検索するか、計算が必要かどうか** は人が決めていました。
エージェントでは、それを LLM 自身に決めさせます。

---

# Part 2: ツール呼び出しの正体を見る

## 2-1. ツールを定義する

LangChain では、関数に `@tool` を付けるとツールになります。
**関数名・引数の型・docstring（説明文）** が、そのまま LLM に渡される「ツールの説明書」になります。

まず、Python を実行するツールを 1 つだけ作ります。

In [ ]:
# 実行時間: 数秒
@tool
def python_exec(code: str) -> str:
    """Python のコードを実行し、print() で表示された内容を返す。計算をするときに使う。"""
    return _run_python(code, WORKSPACE)


schema = convert_to_openai_tool(python_exec)
print(json.dumps(schema, ensure_ascii=False, indent=2))

## 2-2. ツールの説明書は、プロンプトに書き込まれる

ツールの説明書（上の JSON）は、チャットテンプレートによって **プロンプトの中に文章として** 書き込まれます。
実際にモデルに渡される文字列を見てみます。

In [ ]:
# 実行時間: 数秒
messages = [{'role': 'user', 'content': Q_CALC}]
prompt = tokenizer.apply_chat_template(messages, tools=[schema], add_generation_prompt=True, tokenize=False)
print(prompt)

`<tools>` の中にツールの説明書があり、その下に「ツールを呼ぶときは `<tool_call>` の中に JSON を書け」という指示があります。
Qwen2.5 は、この形式でツールを呼ぶように学習されています。

## 2-3. モデルは「ツールを呼びたい」と文章で書く

このプロンプトの続きを書かせます。

In [ ]:
# 実行時間: 約20秒
inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=300, do_sample=False)
raw = tokenizer.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=False)
print(raw)

モデルの出力は、ただの文字列です。`<tool_call>` の中に、呼びたいツールの名前と引数が JSON で書かれているはずです。
**モデル自身はコードを実行していません。** 「これを実行してほしい」と書いただけです。

## 2-4. プログラムが読んで実行し、結果を返す

この文字列から JSON を取り出して、実際に関数を実行します。
結果は `tool` の役割のメッセージとして会話に足し、もう一度モデルに続きを書かせます。

In [ ]:
# 実行時間: 約30秒
content, tool_calls, invalid = parse_tool_calls(raw)
print(f'取り出したツール呼び出し: {tool_calls}')
if invalid:
    print(f'読み取れなかった呼び出し: {invalid}')

messages.append({'role': 'assistant', 'content': content,
                 'tool_calls': [{'type': 'function', 'function': {'name': tc['name'], 'arguments': tc['args']}}
                                for tc in tool_calls]})
for tc in tool_calls:
    result = python_exec.invoke(tc['args'])          # ← ここで初めて、コードが実際に実行される
    print(f'\n実行結果: {result}')
    messages.append({'role': 'tool', 'content': result, 'name': tc['name']})

prompt2 = tokenizer.apply_chat_template(messages, tools=[schema], add_generation_prompt=True, tokenize=False)
inputs = tokenizer(prompt2, return_tensors='pt', add_special_tokens=False).to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
print(f'\n最終回答: {tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)}')

これが **ReAct**（Reasoning + Acting）の 1 周です。

```
考える（Reason）   LLM が「python_exec を、このコードで呼びたい」と書く
行動する（Act）    プログラムがツールを実行する
観察する（Observe） 実行結果を会話に足し、LLM に続きを書かせる
                   → もう一度ツールを呼ぶか、答えを書く
```

ツールを呼ぶかどうか、どのツールを、どんな引数で呼ぶかを決めているのは LLM です。
一方で、**実際に実行するかどうかを決めているのはプログラム** です。危ない操作を止めたり、確認を挟んだりできるのは、この境目があるからです。

---

# Part 3: LangChain のチャットモデルにまとめる

Part 2 の「プロンプトにツールを書き込む → 生成する → `<tool_call>` を読み取る」を、
`agent_kit.LocalChatModel` にまとめてあります（中身は Part 2 と同じです。`chapter3/agent_kit.py` を開いて確かめられます）。

`bind_tools()` でツールを渡すと、`invoke()` の戻り値（`AIMessage`）の `tool_calls` に、読み取ったツール呼び出しが入ります。

In [ ]:
# 実行時間: 約20秒
llm = LocalChatModel(model=model, tokenizer=tokenizer)
ai = llm.bind_tools([python_exec]).invoke([HumanMessage(Q_CALC)])

print(f'content   : {ai.content!r}')
print(f'tool_calls: {ai.tool_calls}')
print(f'トークン数: {ai.usage_metadata}')

---

# Part 4: LangGraph でループを組み立てる

Part 2 では、ループを手で 1 周だけ回しました。これを、ツール呼び出しがなくなるまで繰り返すようにします。

LangGraph では、処理を **ノード**（箱）と **エッジ**（矢印）の **グラフ** として書きます。

| 部品 | 役割 |
|---|---|
| `MessagesState` | グラフの中を流れる **状態**。ここでは会話のメッセージのリスト。各ノードは、足したいメッセージを返す |
| `agent` ノード | 考える。LLM が、次にツールを呼ぶか、答えるかを決める |
| `tools` ノード | 行動する。`ToolNode` が、直前の `tool_calls` を実行し、結果を `ToolMessage` として足す |
| `tools_condition` | 分かれ道。直前のメッセージに `tool_calls` があれば `tools` へ、なければ終了（`END`）へ |

まず、ツールを 5 つ用意します。

In [ ]:
# 実行時間: 約20秒
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 第2章ハンズオン③と同じ検索の仕組み（検索対象: chapter2/data/rag_docs）
docs_dir = CH3_DIR.parent / 'chapter2' / 'data' / 'rag_docs'
documents = [Document(page_content=p.read_text(encoding='utf-8'), metadata={'source': p.name})
             for p in sorted(docs_dir.glob('*.md')) if p.name != 'README.md']
chunks = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer, chunk_size=256, chunk_overlap=32, separators=['\n\n', '\n', '。', '、', ' ', '']).split_documents(documents)
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
                                   model_kwargs={'device': device}, encode_kwargs={'normalize_embeddings': True})
vectorstore = Chroma(collection_name='ch3-docs', embedding_function=embeddings,
                     collection_metadata={'hnsw:space': 'cosine'})
vectorstore.reset_collection()
vectorstore.add_documents(chunks)


@tool
def search_docs(query: str) -> str:
    """このハンズオンの資料（参加者手順書・機械学習の解説）を検索し、関係のある部分を返す。
    ハンズオンの決まりや手順、機械学習の用語について調べるときに使う。"""
    docs = vectorstore.similarity_search(query, k=3)
    return '\n\n---\n\n'.join(f'[出典: {d.metadata["source"]}]\n{d.page_content}' for d in docs)


@tool
def run_python(code: str) -> str:
    """Python のコードを実行し、print() で表示された内容を返す。計算や集計をするときに使う。
    結果は必ず print() で表示すること。"""
    return _run_python(code, WORKSPACE)


@tool
def write_file(filename: str, content: str) -> str:
    """作業ディレクトリにテキストファイルを書き込む（同じ名前のファイルは上書きする）。"""
    path = workspace_path(WORKSPACE, filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
    return f'{filename} に {len(content)} 文字を書き込みました'


@tool
def read_file(filename: str) -> str:
    """作業ディレクトリのテキストファイルを読み、中身を返す。"""
    path = workspace_path(WORKSPACE, filename)
    if not path.exists():
        return f'エラー: {filename} は存在しません。list_files で一覧を確認してください'
    return path.read_text(encoding='utf-8')[:3000]


@tool
def list_files() -> str:
    """作業ディレクトリにあるファイルの一覧を返す。"""
    files = sorted(str(p.relative_to(WORKSPACE)) for p in WORKSPACE.rglob('*') if p.is_file())
    return '\n'.join(files) if files else '（ファイルはありません）'


TOOL_LIST = [search_docs, run_python, write_file, read_file, list_files]
print('ツール:', ', '.join(t.name for t in TOOL_LIST))

In [ ]:
# 実行時間: 数秒
SYSTEM_PROMPT = (
    'あなたは、ML ハンズオンの参加者を手伝うアシスタントです。'
    'ハンズオンの決まりや用語は search_docs で調べ、計算は run_python で行い、推測で答えないでください。'
    '日本語で、簡潔に答えてください。'
)


llm = LocalChatModel(model=model, tokenizer=tokenizer, max_new_tokens=512)
llm_with_tools = llm.bind_tools(TOOL_LIST)


def agent_node(state: MessagesState) -> dict:
    """考える: これまでの会話を読み、次にツールを呼ぶか、答えるかを決める"""
    messages = [SystemMessage(SYSTEM_PROMPT)] + state['messages']
    return {'messages': [llm_with_tools.invoke(messages)]}


builder = StateGraph(MessagesState)
builder.add_node('agent', agent_node)              # 考える
builder.add_node('tools', ToolNode(TOOL_LIST))     # 行動する（ツールを実行して、結果をメッセージに足す）
builder.add_edge(START, 'agent')
builder.add_conditional_edges('agent', tools_condition)   # ツール呼び出しがあれば tools へ、なければ終了（END）へ
builder.add_edge('tools', 'agent')                 # 観察した結果を持って、また考える
graph = builder.compile()

print(graph.get_graph().draw_ascii())

図の実線は必ず通る道、点線は条件で分かれる道です。
`agent` から点線が 2 本（`tools` と `__end__`）出ていて、`tools` からは `agent` に戻る実線が出ています。
**エージェントが「何を決めているか」は、この点線の分かれ道だけ** です。

---

# Part 5: エージェントを動かす

`graph.stream()` を使うと、ノードを 1 つ通るごとに状態を受け取れます。新しく足されたメッセージを順に表示します。

In [ ]:
# 実行時間: 数秒
def show(message):
    """メッセージを 1 件ずつ、種類が分かるように表示する"""
    if message.type == 'human':
        print(f'👤 ユーザー: {message.content}')
    elif message.type == 'ai' and message.tool_calls:
        if message.content:
            print(f'🤔 考え: {message.content}')
        for tc in message.tool_calls:
            print(f'🔧 ツール呼び出し: {tc["name"]}({json.dumps(tc["args"], ensure_ascii=False)[:200]})')
    elif message.type == 'ai':
        print(f'💬 回答: {message.content}')
    elif message.type == 'tool':
        print(f'📄 ツールの結果（{message.name}）: {str(message.content)[:200]}')
    print()


def run_agent(question: str, recursion_limit: int = 12, callbacks=None) -> list:
    """エージェントに質問し、途中経過を表示しながら最後の状態のメッセージを返す"""
    config = {'recursion_limit': recursion_limit, 'callbacks': callbacks or []}
    shown = 0
    for state in graph.stream({'messages': [HumanMessage(question)]}, config=config, stream_mode='values'):
        for m in state['messages'][shown:]:
            show(m)
        shown = len(state['messages'])
    return state['messages']

## 5-1. ハンズオンの決まりを聞く → 検索する

In [ ]:
# 実行時間: 約1分
trace_rule = TraceHandler()
_ = run_agent(Q_RULE, callbacks=[trace_rule])

## 5-2. 計算を頼む → Python を書いて実行する

In [ ]:
# 実行時間: 約1分
trace_calc = TraceHandler()
_ = run_agent(Q_CALC, callbacks=[trace_calc])

## 5-3. 調べてまとめ、ファイルに保存してもらう → 複数のツールを順に使う

In [ ]:
# 実行時間: 約2分
Q_TASK = ('参加者手順書で、OutOfMemoryError が出たときの対処を調べ、'
          '3 行の箇条書きにまとめて notes/oom.md に保存してください。')
trace_task = TraceHandler()
_ = run_agent(Q_TASK, callbacks=[trace_task])

print('=== 作業ディレクトリ ===')
print(list_files.invoke({}))
if (WORKSPACE / 'notes' / 'oom.md').exists():
    print('\n=== notes/oom.md ===')
    print((WORKSPACE / 'notes' / 'oom.md').read_text(encoding='utf-8'))

エージェントは、検索 → 書き込み のように、ツールを **自分で選んで順に** 使ったはずです。
同じ質問でも、モデルや言い回しによって、使うツールの順番や回数は変わります。うまくいかなかったときは、次の Part のトレースで原因を探します。

---

# Part 6: トレースを読む — 観測性とコスト

エージェントは、1 つの質問のために LLM を何度も呼びます。
**何回呼んだか、何トークン使ったか、どこで時間がかかったか** を記録しておかないと、遅い・高い・間違えたときに原因が分かりません。

`TraceHandler` は、LangChain のコールバックで LLM とツールの呼び出しを 1 件ずつ記録します。

In [ ]:
# 実行時間: 数秒
print('=== 5-3 のトレース ===')
trace_task.print_table()

表の読み方は次のとおりです。

- **入力トークン数は、呼ぶたびに増えていく**: LLM は毎回、システムプロンプト・ツールの説明書・それまでの会話をすべて読み直します。ツールの結果が長いと、次の呼び出しの入力が一気に増えます
- **コストは入力トークン数でほぼ決まる**: API の LLM は、トークン数に比例して料金がかかります。ループが長いエージェントほど高くなります
- **時間の大半は LLM**: ツール（検索・Python）は一瞬で終わり、待ち時間のほとんどは LLM の生成です

3 つの質問のコストを並べます。

In [ ]:
# 実行時間: 数秒
print(f'{"質問":<8} {"LLM回数":>7} {"ツール回数":>9} {"入力tok":>8} {"出力tok":>8} {"LLM秒":>6}')
for name, t in [('決まり', trace_rule), ('計算', trace_calc), ('調べて保存', trace_task)]:
    s = t.totals()
    print(f'{name:<8} {s["LLM の呼び出し回数"]:>7} {s["ツールの呼び出し回数"]:>9} {s["入力トークン数"]:>8} '
          f'{s["出力トークン数"]:>8} {s["LLM の時間（秒）"]:>6}')

## 6-2. ループに上限を付ける

エージェントが同じツールを何度も呼び続けると、時間もコストも際限なく増えます。
LangGraph では `recursion_limit` で、ノードを通る回数の上限を決められます。上限に達すると `GraphRecursionError` で止まります。

わざと上限を小さくして、止まる様子を見ます。

In [ ]:
# 実行時間: 約30秒
from langgraph.errors import GraphRecursionError

try:
    run_agent(Q_TASK, recursion_limit=2)
    print('上限に達する前に終わりました（ツールを使わずに答えた可能性があります）')
except GraphRecursionError as e:
    print(f'⛔ 上限で止まりました: {type(e).__name__}')

上限 2 では、`agent` → `tools` を 1 回通っただけで止まります。
実務では「普通なら 5 回で終わる仕事なら上限を 10〜20 にする」のように、**想定の数倍** を上限にします。上限は、暴走したときの安全装置です。

---

## まとめ

1. エージェントは **LLM + ループ + ツール**。LLM は「ツールを呼びたい」と決まった形式（`<tool_call>` の JSON）で書くだけで、実行するのはプログラム
2. ツールの **名前・引数の型・説明文** が、そのまま LLM への説明書になる。説明文の書き方で、LLM がツールを選べるかが変わる
3. LangGraph では、考える（agent）→ 行動する（tools）のループを **グラフ** として書く。エージェントの判断は、`tools_condition` の分かれ道に集約される
4. **トレース** で、LLM の呼び出し回数・トークン数・時間を記録する。入力トークン数は呼ぶたびに増える。ループには上限を付ける

次のハンズオン②では、気象 API や社内データベースのような「外部のシステム」をツールとして登録します。